In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

def get_page(url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
        'Accept-Language': 'zh-CN,zh;q=0.8,zh-TW;q=0.7,zh-HK;q=0.5,en-US;q=0.3,en;q=0.2',
        'Referer': 'https://movie.douban.com/',
        # 注意：如果你没有cookie，可以注释掉下面这行，但可能会遇到反爬。
        'Cookie': 'bid=h1KkEOxg35I; _pk_id.100001.4cf6=b0bea3b8629c0ba3.1781253371.; __utmz=30149280.1781253371.1.1.utmcsr=(direct)|utmccn=(direct)|utmcmd=(none); __utmz=223695111.1781253371.1.1.utmcsr=(direct)|utmccn=(direct)|utmcmd=(none); __yadk_uid=DO9MRAeOZk0aWfCfUe7nZLWu1WR9sUWe; _pk_ses.100001.4cf6=1; ap_v=0,6.0; __utma=30149280.2115076673.1781253371.1781253371.1781333943.2; __utmb=30149280.0.10.1781333943; __utmc=30149280; __utma=223695111.444196898.1781253371.1781253371.1781333943.2; __utmb=223695111.0.10.1781333943; __utmc=223695111'
    }
    
    try:
        r = requests.get(url, headers=headers, timeout=15)
        r.raise_for_status()
        r.encoding = 'utf-8'
        # 调试：打印前500字符，检查是否包含电影信息
        # print(r.text[:500])
        return r.text
    except Exception as e:
        print(f'请求失败: {e}')
        return None

def parse_page(html):
    soup = BeautifulSoup(html, 'html.parser')
    # 第一步：找到所有电影条目所在的容器
    movie_items = soup.find_all('div', class_='item')
    print(f"在页面中找到 {len(movie_items)} 个电影条目")

    movie_list = []

    for item in movie_items:
        try:
            # 提取电影标题
            title_tag = item.find('span', class_='title')
            if not title_tag:
                continue
            title = title_tag.text

            # 提取评分
            rating_tag = item.find('span', class_='rating_num')
            if not rating_tag:
                continue
            rating = rating_tag.text



            # 获取所有 <span>，最后一个通常是评价人数
        
            evaluate_span = item.find('span', string=re.compile(r'\d+人评价'))
            if evaluate_span:
                 evaluate_num = int(re.search(r'(\d+)', evaluate_span.get_text(strip=True)).group(1))
            else:
                evaluate_num = 0

            # 从信息段落中提取年份（这是另一种策略，不依赖单个容器）
            info_ps = item.find('div', class_='bd').find_all('p')
            if info_ps:
                info = info_ps[0].text.strip()
                year_match = re.search(r'(\d{4})', info)
                year = year_match.group(1) if year_match else ''
            else:
                year = ''

            movie_list.append({
                'title': title,
                'rating': float(rating),
                'evaluate_num': int(evaluate_num),
                'year': year
            })
        except Exception as e:
            print(f"解析单条记录出错，已跳过。错误信息：{e}")
            continue
    return movie_list





def main():
    base_url = 'https://movie.douban.com/top250?start={}'
    all_movies = []

    for start in range(0, 50, 25):
        url = base_url.format(start)
        print(f'正在爬取: {url}')
        html = get_page(url)
        if html:
            movies = parse_page(html)
            print(f"本页解析到 {len(movies)} 部电影")
            all_movies.extend(movies)
            time.sleep(5)
        else:
            print(f'第{start//25+1}页爬取失败')
            break

    if all_movies:
        df = pd.DataFrame(all_movies)
        df.to_csv('douban_top250.csv', index=False, encoding='utf-8-sig')
        print(f'爬取完成，共 {len(all_movies)} 条记录，已保存到 douban_top250.csv')
    else:
        print("没有获取到任何数据，请检查网络或豆瓣反爬策略。")

if __name__ == '__main__':
    main()
